# XGBoost on Morgan Fingerprints (Representation B)

This notebook trains and evaluates an **XGBoost classifier** using **Morgan fingerprints** as the molecular representation.

We focus here only on **Representation B (fingerprints)** for clarity.


### Import Required Libraries

- **numpy**: Load fingerprint matrices and labels
- **xgboost**: Gradient boosting model (XGBClassifier)
- **sklearn (KFold, metrics)**: Cross-validation and evaluation
- **warnings**: Suppress noisy warnings


In [1]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score

# XGBoost
try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError(
        "xgboost is not installed in this environment. Install it with `pip install xgboost` or `conda install -c conda-forge xgboost`."
    )

import warnings
warnings.filterwarnings("ignore")

seed = 20231124
np.random.seed(seed)


## Step 1: Load Fingerprint Data

Load the fingerprint matrix and labels generated in `1_get_fingerprints.ipynb`:
- `training_fingerprints_matrix.npy` → feature matrix \(X\)
- `training_fingerprints_labels.npy` → labels \(y\) (0/1)


In [2]:
# Load fingerprint matrix (X) and labels (y)
X_fp = np.load("training_fingerprints_matrix.npy")
y_fp = np.load("training_fingerprints_labels.npy")

print(f"Fingerprint matrix shape: {X_fp.shape}")
print(f"Labels shape: {y_fp.shape}")
print(f"Class balance: {(y_fp==1).sum()} active / {(y_fp==0).sum()} inactive")


Fingerprint matrix shape: (202895, 2048)
Labels shape: (202895,)
Class balance: 12514 active / 190381 inactive


## Step 2: Define XGBoost Model and Cross-Validation

Evaluation:
- **10-fold cross-validation** (same as Random Forest notebooks)
- Metrics: **Accuracy** and **ROC AUC** per fold + mean values.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

seed = 42

kf_fp = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)

xgb_acc_fp = []
xgb_auc_fp = []

params_fp_A = {
    "n_estimators": 600,
    "learning_rate": 0.05,
    "max_depth": 6,          # ↑ djupare träd
    "subsample": 0.9,
    "colsample_bytree": 0.7, # ↑ ser fler features per träd
    "reg_alpha": 0.3,        # mindre L1
    "reg_lambda": 1.0,       # mindre L2
    "min_child_weight": 1,
    "gamma": 0.0,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    # Lägg till om obalanserat:
    # "scale_pos_weight": scale_pos_weight,
}


fold = 0
for train_index, test_index in kf_fp.split(X_fp, y_fp):

    # For NumPy arrays:
    X_train_fp, X_test_fp = X_fp[train_index], X_fp[test_index]
    y_train_fp, y_test_fp = y_fp[train_index], y_fp[test_index]

    model_xgb = XGBClassifier(**params_fp_A)

    model_xgb.fit(X_train_fp, y_train_fp)

    # Predictions
    preds_fp = model_xgb.predict(X_test_fp)
    proba_fp = model_xgb.predict_proba(X_test_fp)[:, 1]

    # Metrics
    acc_fp = accuracy_score(y_test_fp, preds_fp)
    auc_fp = roc_auc_score(y_test_fp, proba_fp)

    xgb_acc_fp.append(acc_fp)
    xgb_auc_fp.append(auc_fp)

    fold += 1
    print(f"Done with fold {fold}/10")

print("\nXGBoost on Fingerprints (Representation B)")
print("Accuracy per fold:", xgb_acc_fp)
print("Mean CV accuracy:", sum(xgb_acc_fp) / len(xgb_acc_fp))
print("AUC per fold:", xgb_auc_fp)
print("Mean CV AUC:", sum(xgb_auc_fp) / len(xgb_auc_fp))


NameError: name 'params' is not defined